# PoS tagging e morfologia italiani con BERT XXL, ELECTRA XXL e GePpeTto

Due ablation separate: originale contro minuscolo per UPOS (18 run), poi UPOS contro UPOS+FEATS sul solo testo originale (9 nuovi run). La seconda fase continua dal checkpoint UPOS corrispondente a modello e seed. Selezionare `TASK_STAGE` nella configurazione; i 18 run UPOS già salvati vengono riutilizzati.


In [1]:
%pip install -q "transformers==4.57.1" "datasets==4.8.5" "accelerate==1.11.0" "safetensors>=0.5,<1" "numpy>=2,<3" "scikit-learn==1.8.0" "pandas>=2,<4"


Note: you may need to restart the kernel to use updated packages.


## 2. Configuration


In [ ]:
import csv, gc, hashlib, importlib.metadata, json, math, platform, re, statistics, time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, cast
# Load pandas/PyArrow before PyTorch to avoid the Windows native import conflict.
from datasets import Dataset, load_dataset
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset as TorchDataset
from IPython.display import display
from transformers import (AutoConfig, AutoModel, AutoTokenizer, GPT2ForTokenClassification,
                          PreTrainedModel, Trainer, TrainingArguments, set_seed)
from transformers.modeling_outputs import TokenClassifierOutput

TASK_STAGE = "upos_feats"  # "upos" or "upos_feats"; the second stage reuses the matching UPOS run.
FEATS_WEIGHT = 1.0
NONE = "<NONE>"
if TASK_STAGE not in {"upos", "upos_feats"}:
    raise ValueError("Unknown task stage")

SEEDS = [42, 123, 456]
VARIANTS = ["original", "lowercase"]
UPOS_TAGS = ("ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN", "NUM",
             "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "SYM", "VERB", "X")
LABEL_IDS = {tag: i for i, tag in enumerate(UPOS_TAGS)}

DATA = {
    "repo_id": "universal-dependencies/universal_dependencies",
    "config": "it_isdt",
    "ud_version": "2.15",
    "release_tag": "ud2.15-loader-v2.3.0",
    "revision": "8426077f2e778f20d46a5b89a187e4f991cfe6c4",
    "prepared_dir": "data/prepared",
    "parquet_sha256": {
        "train": "7cf4df5e42e54335c1a649a248cb3d2aa85876f22a4ae1c32b60202a30d2543d",
        "dev": "9a15c35ba8d307c9d623eddacff6f759a25d7f52ed575f2bbe2d796216ef6e1f",
        "test": "ce4b6b696417174934c8a39da6bb88e1894689e8fa1bb2b87fe108a123a1bc15"
    }
}

MODELS = {
    "bert": {
        "kind": "encoder",
        "repo_id": "dbmdz/bert-base-italian-xxl-cased",
        "revision": "9bdf84aedf00aa4e99b01a6f2c76f6070ac7240b",
        "epochs": 10,
        "learning_rate": 2e-05,
        "batch_size": 8,
        "eval_batch_size": 16,
        "gradient_accumulation": 1,
        "max_length": 512,
        "dropout": 0.1,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1
    },
    "electra": {
        "kind": "encoder",
        "repo_id": "dbmdz/electra-base-italian-xxl-cased-discriminator",
        "revision": "c6d38915b86f51db0ba5d2a2c150b01c2289c8e7",
        "epochs": 10,
        "learning_rate": 2e-05,
        "batch_size": 8,
        "eval_batch_size": 16,
        "gradient_accumulation": 1,
        "max_length": 512,
        "dropout": 0.1,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1
    },
    "geppetto": {
        "kind": "decoder",
        "tagger_version": "gpt2_upos_mean_pooling_v1",
        "repo_id": "LorenzoDeMattei/GePpeTto",
        "revision": "d4b79e7cb2b41925adf2837a7715243f56eaf705",
        "epochs": 10,
        "learning_rate": 2e-05,
        "batch_size": 8,
        "eval_batch_size": 16,
        "gradient_accumulation": 1,
        "max_length": 512,
        "dropout": 0.1,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1
    }
}

PROJECT = Path.cwd().resolve()
for candidate in (PROJECT, PROJECT.parent, PROJECT / "italian-pos-tagging"):
    if (candidate / "notebooks/Italian_UPOS_AI.ipynb").is_file():
        PROJECT = candidate
        break
else:
    raise RuntimeError("Aprire il notebook dalla nuova cartella del progetto o da notebooks/")

UPOS_EXPERIMENT = {
    "schema": "upos_mean_pooling_trainer_v1", "dataset": DATA,
    "models": MODELS, "seeds": SEEDS, "variants": VARIANTS,
    "pooling": "mean", "selection_metric": "upos_macro_f1_all",
    "output_dir": "outputs/upos_trainer_v1",
}
SYSTEMS = {f"{model}_{variant}": {
    **spec, "model": model, "variant": variant, "name": f"{model.upper()} / {variant}",
} for model, spec in MODELS.items() for variant in VARIANTS}
ACTIVE_SYSTEMS = list(SYSTEMS)
UPOS_EXPERIMENT["active_systems"] = ACTIVE_SYSTEMS
UPOS_EXPERIMENT["trainer"] = {"eval_strategy": "epoch", "save_strategy": "epoch",
    "load_best_model_at_end": True, "save_total_limit": 1, "disable_tqdm": False}
UPOS_DIR = PROJECT / UPOS_EXPERIMENT["output_dir"]
EXPERIMENT = json.loads(json.dumps(UPOS_EXPERIMENT))
if TASK_STAGE == "upos_feats":
    # Separate studies: case ablation for UPOS; FEATS extension on original text only.
    VARIANTS = ["original"]
    ACTIVE_SYSTEMS = [system for system in SYSTEMS if SYSTEMS[system]["variant"] == "original"]
    EXPERIMENT.update(schema="upos_feats_mean_pooling_trainer_v1", task_stage=TASK_STAGE,
        output_dir="outputs/upos_feats_trainer_v1", parent_output_dir=UPOS_EXPERIMENT["output_dir"],
        selection_metric="joint_macro_f1", feats_weight=FEATS_WEIGHT,
        variants=VARIANTS, active_systems=ACTIVE_SYSTEMS)
OUTPUT_DIR = EXPERIMENT_DIR = PROJECT / EXPERIMENT["output_dir"]
print("Device:", torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU")


## 3. Data


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def require_revision(revision):
    if not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise ValueError("Use an immutable 40-character Hugging Face commit SHA")


def transform_tokens(tokens, variant):
    if variant == "original":
        return list(tokens)
    if variant == "lowercase":
        return [word.lower() for word in tokens]
    raise ValueError(f"Unknown input variant: {variant}")


def normalize_sentence(row, index, label_names=None):
    tokens = list(row["tokens"])
    tags = list(row["upos"])
    if label_names is not None:
        tags = [label_names[tag] if isinstance(tag, int) else tag for tag in tags]
    if not tokens or len(tokens) != len(tags):
        raise ValueError(f"Invalid word/tag alignment in sentence {index}")
    if any(not isinstance(word, str) or not word for word in tokens):
        raise ValueError(f"Invalid word form in sentence {index}")
    if set(tags) - set(UPOS_TAGS):
        raise ValueError(f"Non-UPOS annotation in sentence {index}; require syntactic words only")
    return {"sent_id": str(row.get("sent_id", row.get("idx", index))), "tokens": tokens, "upos": tags}


def prepare_corpus(config, project, local_parquet=None):

    spec = config["dataset"]
    require_revision(spec["revision"])
    target = Path(project) / spec["prepared_dir"]
    manifest_path = target / "manifest.json"
    if manifest_path.exists():
        load_corpus(config, project)
        return json.loads(manifest_path.read_text(encoding="utf-8"))
    if target.exists() and any(target.iterdir()):
        raise RuntimeError("Incomplete prepared corpus: use a new prepared_dir or restore its files")
    if local_parquet is None:
        dataset = load_dataset(spec["repo_id"], spec["config"], revision=spec["revision"], cache_dir=str(Path(project) / "data/hf-cache"))
    else:
        local_parquet = Path(local_parquet)
        if not local_parquet.is_absolute():
            local_parquet = Path(project) / local_parquet
        files = {split: local_parquet / f"{split}.parquet" for split in ("train", "dev", "test")}
        expected = spec.get("parquet_sha256", {})
        for split, path in files.items():
            if split not in expected or sha256(path) != expected[split]:
                raise ValueError(f"Local Parquet does not match the pinned Hugging Face snapshot: {split}")
        dataset = load_dataset("parquet", data_files={split: str(path) for split, path in files.items()}, cache_dir=str(Path(project) / "data/hf-cache"))
    split_names = {"train": "train", "dev": "dev" if "dev" in dataset else "validation", "test": "test"}
    corpus = {}
    for split, source in split_names.items():
        table = dataset[source]
        feature = table.features["upos"]
        label_names = getattr(getattr(feature, "feature", None), "names", None)
        corpus[split] = [normalize_sentence(row, i, label_names) for i, row in enumerate(table)]
    if set(tag for row in corpus["train"] for tag in row["upos"]) != set(UPOS_TAGS):
        raise ValueError("Training corpus does not match the expected 17-tag UPOS inventory")
    target.mkdir(parents=True, exist_ok=True)
    manifest = {"dataset": spec, "labels": list(UPOS_TAGS), "splits": {}}
    for split, rows in corpus.items():
        path = target / f"{split}.jsonl"
        path.write_text("".join(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n" for row in rows), encoding="utf-8")
        manifest["splits"][split] = {"sentences": len(rows), "words": sum(len(row["tokens"]) for row in rows), "sha256": sha256(path)}
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return manifest


def load_corpus(config, project):
    target = Path(project) / config["dataset"]["prepared_dir"]
    manifest = json.loads((target / "manifest.json").read_text(encoding="utf-8"))
    if manifest["dataset"] != config["dataset"] or manifest["labels"] != list(UPOS_TAGS):
        raise ValueError("Prepared corpus and configuration differ; use another prepared_dir")
    corpus = {}
    for split, details in manifest["splits"].items():
        path = target / f"{split}.jsonl"
        if sha256(path) != details["sha256"]:
            raise ValueError(f"Prepared corpus hash mismatch: {split}")
        corpus[split] = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
        if len(corpus[split]) != details["sentences"] or sum(len(row["tokens"]) for row in corpus[split]) != details["words"]:
            raise ValueError(f"Prepared corpus count mismatch: {split}")
    return corpus, manifest


local_parquet = PROJECT / "data/hf-parquet"
local_available = all((local_parquet / f"{split}.parquet").is_file() for split in ("train", "dev", "test"))
prepare_corpus(EXPERIMENT, PROJECT, local_parquet if local_available else None)
sentences, data_manifest = load_corpus(EXPERIMENT, PROJECT)
for split, details in data_manifest["splits"].items():
    print(f"{split}: {details['sentences']:,} frasi, {details['words']:,} parole")

display(pd.DataFrame([{"Split": split, "Sentences": len(rows),
    "Words": sum(len(row["tokens"]) for row in rows)} for split, rows in sentences.items()]))


def canonical_feats(value):
    if value is None or value == "_" or value == "":
        return {}
    if not isinstance(value, str):
        raise ValueError("Expected FEATS in CoNLL-U string format")
    result = {}
    for item in value.split("|"):
        key, separator, values = item.partition("=")
        if not separator or not key or not values or key in result:
            raise ValueError(f"Invalid FEATS annotation: {value}")
        result[key] = ",".join(sorted(values.split(",")))
    return result


def load_morphology(corpus):
    files = {split: PROJECT / "data/hf-parquet" / f"{split}.parquet" for split in corpus}
    if all(path.is_file() for path in files.values()):
        for split, path in files.items():
            if sha256(path) != DATA["parquet_sha256"][split]:
                raise ValueError(f"FEATS source checksum mismatch: {split}")
        raw = load_dataset("parquet", data_files={split: str(path) for split, path in files.items()},
                           cache_dir=str(PROJECT / "data/hf-cache"))
    else:
        raw = load_dataset(DATA["repo_id"], DATA["config"], revision=DATA["revision"],
                           cache_dir=str(PROJECT / "data/hf-cache"))
    annotated, hashes = {}, {}
    for split, rows in corpus.items():
        source = raw[split if split in raw else "validation"]
        if len(source) != len(rows):
            raise ValueError(f"FEATS sentence count mismatch: {split}")
        names = getattr(getattr(source.features["upos"], "feature", None), "names", None)
        annotated[split] = []
        for index, (saved, original) in enumerate(zip(rows, source)):
            if normalize_sentence(original, index, names) != saved:
                raise ValueError(f"FEATS/UPOS corpus alignment mismatch: {split}, {index}")
            values = original["feats"]
            if len(values) != len(saved["tokens"]):
                raise ValueError(f"FEATS word count mismatch: {split}, {index}")
            annotated[split].append({**saved, "feats": [canonical_feats(value) for value in values]})
        content = json.dumps([row["feats"] for row in annotated[split]], ensure_ascii=False, sort_keys=True)
        hashes[split] = hashlib.sha256(content.encode("utf-8")).hexdigest()
    return annotated, hashes


feature_names, feature_labels, feature_label_ids, training_pairs = [], {}, {}, set()
if TASK_STAGE == "upos_feats":
    sentences, morphology_hashes = load_morphology(sentences)
    feature_names = sorted({key for row in sentences["train"] for values in row["feats"] for key in values})
    feature_labels = {key: [NONE, *sorted({values[key] for row in sentences["train"]
        for values in row["feats"] if key in values})] for key in feature_names}
    feature_label_ids = {key: {value: index for index, value in enumerate(values)}
                         for key, values in feature_labels.items()}
    training_pairs = {f"{key}={value}" for key, values in feature_labels.items() for value in values if value != NONE}
    EXPERIMENT["morphology"] = {"head_labels": feature_labels, "annotation_sha256": morphology_hashes,
                               "none_label": NONE, "vocabulary_source": "train"}
    display(pd.DataFrame([{"Attribute": key, "Classes": len(values), "Labels": ", ".join(values)}
                          for key, values in feature_labels.items()]))
    for split in ("dev", "test"):
        observed = {f"{key}={value}" for row in sentences[split] for values in row["feats"] for key, value in values.items()}
        print(f"{split}: FEATS pairs absent from training: {sorted(observed - training_pairs)}")
print(f"Stage: {TASK_STAGE}; outputs: {EXPERIMENT_DIR}")


## 4. Tokenization

Tutti i modelli usano il mean pooling delle rappresentazioni delle subword per ottenere un vettore per parola gold. Per una divisione illustrativa `in` + `utile`, usare solo il primo frammento conserverebbe una sola delle rappresentazioni; la media combina anche quelle delle altre parti della parola. Lo stesso vettore alimenta la testa UPOS e, nella seconda fase, le 14 teste FEATS. GePpeTto conserva backbone e testa UPOS della classe ufficiale `GPT2ForTokenClassification`, con pooling per parola aggiunto dal progetto; la sua attenzione resta causale.


In [ ]:
def get_tokenizer(spec):
    options = {"add_prefix_space": True} if spec["kind"] == "decoder" else {}
    tokenizer = AutoTokenizer.from_pretrained(
        spec["repo_id"], revision=spec["revision"], use_fast=True,
        cache_dir=str(PROJECT / "data/hf-cache"), **options,
    )
    if spec["kind"] == "decoder":
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def encode_sentences(sentences, tokenizer, variant, max_length):
    if not tokenizer.is_fast or tokenizer.padding_side != "right":
        raise ValueError("A fast tokenizer with right padding is required for word tagging")
    examples = []
    for index, sentence in enumerate(sentences):
        encoded = tokenizer(transform_tokens(sentence["tokens"], variant), is_split_into_words=True, truncation=False)
        if len(encoded["input_ids"]) > max_length:
            raise ValueError(f"Sentence {sentence['sent_id']} ({index}) exceeds max_length={max_length}; no words were truncated")
        ids = [-1 if word is None else word for word in encoded.word_ids()]
        if set(ids) - {-1} != set(range(len(sentence["tokens"]))):
            raise ValueError(f"Incomplete subword coverage: {sentence['sent_id']}")
        labels = [LABEL_IDS[tag] for tag in sentence["upos"]]
        if TASK_STAGE == "upos_feats":
            labels = [[tag, *[feature_label_ids[key].get(values.get(key, NONE), 0) for key in feature_names]]
                      for tag, values in zip(labels, sentence["feats"])]
        examples.append({**dict(encoded), "word_ids": ids, "labels": labels})
    return examples

class WordCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, rows):
        inputs = [{key: value for key, value in row.items() if key not in {"word_ids", "labels"}} for row in rows]
        batch = self.tokenizer.pad(inputs, padding=True, return_tensors="pt")
        width = batch["input_ids"].shape[1]
        batch["word_ids"] = torch.tensor([row["word_ids"] + [-1] * (width - len(row["word_ids"])) for row in rows])
        counts = [len(row["labels"]) for row in rows]
        batch["word_counts"] = torch.tensor(counts)
        padding = [-100] * (1 + len(feature_names)) if TASK_STAGE == "upos_feats" else -100
        batch["labels"] = torch.tensor([row["labels"] + [padding] * (max(counts) - len(row["labels"])) for row in rows])
        return batch

def prepare_data(examples, spec, tokenizer):
    rows = encode_sentences(examples, tokenizer, spec["variant"], spec["max_length"])
    positions = [(s, w) for s, row in enumerate(examples) for w in range(len(row["tokens"]))]
    return Dataset.from_list(rows), positions, {"sentences": len(examples), "words": len(positions),
        "longest_sentence_tokens": max((len(row["input_ids"]) for row in rows), default=0)}

_prepared_system = None

def prepare_system(system):
    global _prepared_system, tokenizer, prepared, model_data, collate
    if _prepared_system == system:
        return
    spec = SYSTEMS[system]
    tokenizer = get_tokenizer(spec)
    assert tokenizer.is_fast
    collate = WordCollator(tokenizer)
    prepared = {split: prepare_data(rows, spec, tokenizer)
                for split, rows in sentences.items()}
    model_data = {split: values[0] for split, values in prepared.items()}
    _prepared_system = system
    print(f"{spec['name']}")
    display(pd.DataFrame([{"Split": split, **values[2]} for split, values in prepared.items()]))



## 5. Metrics


In [ ]:
def score_tags(gold, predicted):
    if len(gold) != len(predicted) or not gold:
        raise ValueError("Gold and predictions must have equal, nonzero length")
    if set(gold) - set(UPOS_TAGS):
        raise ValueError("Unknown gold UPOS tag")
    if set(predicted) - set(UPOS_TAGS):
        raise ValueError("Unexpected prediction label")
    actual, guessed = Counter(gold), Counter(predicted)
    correct = Counter(a for a, b in zip(gold, predicted) if a == b)
    per_tag = []
    for tag in UPOS_TAGS:
        tp, support, count = correct[tag], actual[tag], guessed[tag]
        per_tag.append({
            "tag": tag, "support": support, "tp": tp, "fp": count - tp,
            "fn": support - tp, "precision": tp / count if count else 0.0,
            "recall": tp / support if support else 0.0,
            "f1": 2 * tp / (support + count) if support + count else 0.0,
        })
    present = [row["f1"] for row in per_tag if row["support"]]
    return {
        "upos_accuracy": sum(correct.values()) / len(gold),
        "upos_macro_f1_all": sum(row["f1"] for row in per_tag) / len(UPOS_TAGS),
        "upos_macro_f1_present": sum(present) / len(present),
        "words": len(gold),
    }, per_tag


def feats_text(values):
    return "|".join(f"{key}={value}" for key, value in sorted(values.items())) or "_"


def feature_scores(gold, predicted):
    if len(gold) != len(predicted) or not gold:
        raise ValueError("Gold and predicted FEATS must cover the same nonzero number of words")
    gold_sets = [{f"{key}={value}" for key, value in row.items()} for row in gold]
    predicted_sets = [{f"{key}={value}" for key, value in row.items()} for row in predicted]
    support = Counter(pair for values in gold_sets for pair in values)
    guesses = Counter(pair for values in predicted_sets for pair in values)
    matches = Counter(pair for a, b in zip(gold_sets, predicted_sets) for pair in a & b)
    rows = []
    for pair in sorted(training_pairs | set(support) | set(guesses)):
        tp, actual, guessed = matches[pair], support[pair], guesses[pair]
        rows.append({"feature": pair, "precision": tp / guessed if guessed else 0.0,
            "recall": tp / actual if actual else 0.0, "f1": 2 * tp / (actual + guessed) if actual + guessed else 0.0,
            "support": actual, "tp": tp, "fp": guessed - tp, "fn": actual - tp})
    return pd.DataFrame(rows)


def attribute_scores(gold, predicted):
    keys = sorted(set(feature_names) | {key for values in gold + predicted for key in values})
    return pd.DataFrame([{"attribute": key, "words": len(gold),
        "accuracy": sum(a.get(key, NONE) == b.get(key, NONE) for a, b in zip(gold, predicted)) / len(gold)}
        for key in keys])


def evaluate_sentences(sentences, predictions, predicted_feats=None):
    if len(sentences) != len(predictions):
        raise ValueError("Sentence count mismatch")
    if predicted_feats is not None and len(predicted_feats) != len(sentences):
        raise ValueError("FEATS sentence count mismatch")
    gold, guessed, rows, gold_feats, guessed_feats = [], [], [], [], []
    for sentence_index, (sentence, tags) in enumerate(zip(sentences, predictions)):
        if len(tags) != len(sentence["tokens"]):
            raise ValueError("Word count mismatch: predictions cannot silently omit words")
        gold.extend(sentence["upos"])
        guessed.extend(tags)
        if predicted_feats is not None:
            values = predicted_feats[sentence_index]
            if len(values) != len(tags):
                raise ValueError("FEATS word count mismatch")
            gold_feats.extend(sentence["feats"])
            guessed_feats.extend(values)
        for word_index, (word, actual, predicted) in enumerate(zip(sentence["tokens"], sentence["upos"], tags)):
            row = {"sentence_index": sentence_index, "sent_id": sentence["sent_id"],
                "word_index": word_index, "word": word, "true_label": actual,
                "predicted_label": predicted, "correct": actual == predicted}
            if predicted_feats is not None:
                expected, observed = sentence["feats"][word_index], values[word_index]
                row.update(true_feats=feats_text(expected), predicted_feats=feats_text(observed),
                           feats_correct=expected == observed, joint_correct=actual == predicted and expected == observed)
            rows.append(row)
    scores, per_tag = score_tags(gold, guessed)
    scores["sentences"] = len(sentences)
    if predicted_feats is not None:
        table = feature_scores(gold_feats, guessed_feats)
        exact = np.array([a == b for a, b in zip(gold_feats, guessed_feats)])
        scores.update(feats_exact_accuracy=float(exact.mean()),
            feats_attribute_accuracy=float(attribute_scores(gold_feats, guessed_feats).accuracy.mean()),
            feats_macro_f1_all=float(table.f1.mean()),
            feats_macro_f1_present=float(table.loc[table.support > 0, "f1"].mean()),
            joint_accuracy=float(np.mean(exact & (np.array(gold) == np.array(guessed)))))
    return scores, per_tag, rows


upos_metric_names = ["upos_accuracy", "upos_macro_f1_all", "upos_macro_f1_present"]
metric_names = upos_metric_names + (["feats_exact_accuracy", "feats_attribute_accuracy",
    "feats_macro_f1_all", "feats_macro_f1_present", "joint_accuracy"] if TASK_STAGE == "upos_feats" else [])


def gold_annotations(split):
    return [tag for row in sentences[split] for tag in row["upos"]]


def decode_output(predictions, label_tensor):
    values = predictions[0] if isinstance(predictions, tuple) else predictions
    joint = label_tensor.ndim == 3
    if values.ndim == label_tensor.ndim + 1:
        values = values.argmax(-1)
    if values.shape != label_tensor.shape or label_tensor.ndim not in (2, 3):
        raise ValueError("Expected one prediction per gold word and task head")
    if joint and label_tensor.shape[-1] != 1 + len(feature_names):
        raise ValueError("FEATS head inventory mismatch")
    upos_rows, feature_rows = [], []
    for row, labels in zip(values, label_tensor):
        valid = labels[:, 0] != -100 if joint else labels != -100
        ids = row[valid]
        upos_ids = ids[:, 0] if joint else ids
        if any(int(i) < 0 or int(i) >= len(UPOS_TAGS) for i in upos_ids):
            raise ValueError("Unknown predicted UPOS ID")
        upos_rows.append([UPOS_TAGS[int(i)] for i in upos_ids])
        if joint:
            features = []
            for word in ids:
                values = {}
                for index, key in enumerate(feature_names, 1):
                    label = int(word[index])
                    if not 0 <= label < len(feature_labels[key]):
                        raise ValueError(f"Unknown predicted FEATS ID: {key}")
                    if label != 0:
                        values[key] = feature_labels[key][label]
                features.append(values)
            feature_rows.append(features)
    return upos_rows, feature_rows if joint else None


def metrics_for(split):
    gold = gold_annotations(split)
    def compute(output):
        predictions, predicted_feats = decode_output(output.predictions, output.label_ids)
        upos_labels = output.label_ids[:, :, 0] if output.label_ids.ndim == 3 else output.label_ids
        expected = np.array([LABEL_IDS[tag] for tag in gold])
        assert np.array_equal(upos_labels[upos_labels != -100], expected), "Gold word order changed"
        scores, _, _ = evaluate_sentences(sentences[split], predictions, predicted_feats)
        if split == "dev" and predicted_feats is not None:
            scores["joint_macro_f1"] = (scores["upos_macro_f1_all"] + scores["feats_macro_f1_present"]) / 2
        return scores
    return compute


def reduce_logits(logits, labels):
    logits = logits[0] if isinstance(logits, tuple) else logits
    return logits.argmax(-1)


## 6. Models


In [ ]:
def mean_pool_words(hidden, word_ids, word_counts, attention_mask=None):
    if hidden.ndim != 3 or word_ids.shape != hidden.shape[:2]:
        raise ValueError("Expected hidden[B,S,H] and word_ids[B,S]")
    if word_counts.shape != (hidden.shape[0],) or not bool((word_counts > 0).all()):
        raise ValueError("Every sentence must contain at least one word")
    max_words = int(word_counts.max())
    valid = word_ids >= 0
    if attention_mask is not None:
        valid = valid & attention_mask.bool()
    if bool((valid & (word_ids >= word_counts[:, None])).any()):
        raise ValueError("A subword points outside its sentence's word inventory")
    # Masked special tokens and padding contribute zero to the word sums.
    indices = word_ids.clamp(min=0, max=max_words - 1)
    sums = hidden.new_zeros((hidden.shape[0], max_words, hidden.shape[-1]))
    sums.scatter_add_(1, indices.unsqueeze(-1).expand_as(hidden), hidden * valid.unsqueeze(-1))
    counts = hidden.new_zeros((hidden.shape[0], max_words))
    counts.scatter_add_(1, indices, valid.to(hidden.dtype))
    word_mask = torch.arange(max_words, device=hidden.device)[None, :] < word_counts[:, None]
    if bool((word_mask & (counts == 0)).any()):
        raise ValueError("At least one gold word has no subword representation")
    return sums / counts.clamp(min=1).unsqueeze(-1), word_mask


def make_feature_heads(config):
    heads = nn.ModuleDict({key: nn.Linear(config.hidden_size, len(values))
                           for key, values in getattr(config, "feats_head_labels", {}).items()})
    for head in heads.values():
        nn.init.normal_(head.weight, std=config.initializer_range)
        nn.init.zeros_(head.bias)
    return heads


def word_tagging_output(model, words, mask, labels):
    hidden = model.dropout(words)
    logits = model.classifier(hidden)
    scores = [logits, *[head(hidden) for head in model.feats_classifiers.values()]]
    joint = len(scores) > 1
    loss = None
    if labels is not None:
        word_labels = labels[:, :, 0] if joint else labels
        if word_labels.shape != mask.shape or not torch.equal(word_labels != -100, mask):
            raise ValueError("Labels must cover every pooled word exactly once")
        if joint and (labels.shape != (*mask.shape, len(scores)) or
                      not torch.equal(labels != -100, mask.unsqueeze(-1).expand_as(labels))):
            raise ValueError("Every FEATS head must cover the same gold words")
        targets = [labels[:, :, index] for index in range(len(scores))] if joint else [labels]
        losses = [nn.functional.cross_entropy(values.reshape(-1, values.shape[-1]), target.reshape(-1), ignore_index=-100)
                  for values, target in zip(scores, targets)]
        loss = losses[0]
        if joint:
            loss = loss + model.config.feats_weight * torch.stack(losses[1:]).mean()
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError("Non-finite loss")
    if joint:
        width = max(values.shape[-1] for values in scores)
        logits = torch.stack([nn.functional.pad(values, (0, width - values.shape[-1]), value=-1e4)
                              for values in scores], dim=2)
    return TokenClassifierOutput(loss=loss, logits=logits)


class EncoderTagger(PreTrainedModel):
    config_class = AutoConfig
    base_model_prefix = "backbone"
    pipeline_version = "upos_mean_pooling_trainer_v1"
    accepts_loss_kwargs = False

    def __init__(self, config, backbone=None):
        self._supports_sdpa = config.model_type == "bert"
        super().__init__(config)
        options = {"add_pooling_layer": False} if config.model_type == "bert" else {}
        self.backbone = backbone if backbone is not None else AutoModel.from_config(config, **options)
        self.dropout = nn.Dropout(config.tagger_dropout)
        self.classifier = nn.Linear(config.hidden_size, len(UPOS_TAGS))
        nn.init.normal_(self.classifier.weight, std=config.initializer_range)
        nn.init.zeros_(self.classifier.bias)
        self.feats_classifiers = make_feature_heads(config)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                word_ids=None, word_counts=None, labels=None):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask,
            token_type_ids=token_type_ids, return_dict=True).last_hidden_state
        words, mask = mean_pool_words(hidden, word_ids, word_counts, attention_mask)
        return word_tagging_output(self, words, mask, labels)


class GPT2WordTagger(GPT2ForTokenClassification):
    # Keep the official GPT-2 backbone, dropout and classification head;
    # aggregate subword hidden states before applying the word-level head.
    pipeline_version = "gpt2_upos_mean_pooling_v1"
    accepts_loss_kwargs = False

    def __init__(self, config):
        super().__init__(config)
        self.feats_classifiers = make_feature_heads(config)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                position_ids=None, word_ids=None, word_counts=None, labels=None):
        output = self.transformer(input_ids=input_ids, attention_mask=attention_mask,
            token_type_ids=token_type_ids, position_ids=position_ids, use_cache=False, return_dict=True)
        words, mask = mean_pool_words(output.last_hidden_state, word_ids, word_counts, attention_mask)
        return word_tagging_output(self, words, mask, labels)


def make_model(system, checkpoint: str | Path | None = None, *, parent: str | Path | None = None):
    spec = SYSTEMS[system]
    tagger_class = EncoderTagger if spec["kind"] == "encoder" else GPT2WordTagger
    expected_features = feature_labels if TASK_STAGE == "upos_feats" else {}
    if parent is not None:
        if checkpoint is not None or TASK_STAGE != "upos_feats":
            raise ValueError("A parent checkpoint is only valid when initializing the joint stage")
        model = tagger_class.from_pretrained(parent)
        assert model.config.pipeline_version == tagger_class.pipeline_version
        assert model.config.id2label == dict(enumerate(UPOS_TAGS))
        assert not model.feats_classifiers, "The parent must be an UPOS-only checkpoint"
        model.config.task_stage = "upos_feats"
        model.config.feats_head_labels = feature_labels
        model.config.feats_weight = FEATS_WEIGHT
        # Added after loading the parent: all pretrained and UPOS weights are retained.
        model.feats_classifiers = make_feature_heads(model.config)
        assert all(parameter.requires_grad for parameter in model.parameters())
        return model
    if TASK_STAGE == "upos_feats" and checkpoint is None:
        raise ValueError("The joint stage requires its matching UPOS parent checkpoint")
    if checkpoint is not None:
        model = tagger_class.from_pretrained(checkpoint)
        assert model.config.pipeline_version == tagger_class.pipeline_version
        assert model.config.id2label == dict(enumerate(UPOS_TAGS))
        assert getattr(model.config, "feats_head_labels", {}) == expected_features
        if expected_features:
            assert model.config.feats_weight == FEATS_WEIGHT
        return model
    if spec["kind"] == "encoder":
        config = AutoConfig.from_pretrained(spec["repo_id"], revision=spec["revision"],
                                           cache_dir=str(PROJECT / "data/hf-cache"))
        config.tagger_dropout = spec["dropout"]
        config.pipeline_version = EncoderTagger.pipeline_version
        config.id2label, config.label2id = dict(enumerate(UPOS_TAGS)), LABEL_IDS
        options = {"add_pooling_layer": False} if config.model_type == "bert" else {}
        backbone = AutoModel.from_pretrained(spec["repo_id"], revision=spec["revision"], config=config,
                                            cache_dir=str(PROJECT / "data/hf-cache"), **options)
        if spec["max_length"] > config.max_position_embeddings:
            raise ValueError("Configured sequence length exceeds encoder positional embeddings")
        model = EncoderTagger(config, backbone)
        assert all(parameter.requires_grad for parameter in model.parameters())
        return model
    if spec["kind"] != "decoder":
        raise ValueError(f"Unknown model kind: {spec['kind']}")
    assert spec["tagger_version"] == GPT2WordTagger.pipeline_version, "Restart the kernel and run all cells in order"
    config = AutoConfig.from_pretrained(spec["repo_id"], revision=spec["revision"],
                                       cache_dir=str(PROJECT / "data/hf-cache"))
    config.id2label, config.label2id = dict(enumerate(UPOS_TAGS)), LABEL_IDS
    config.classifier_dropout = spec["dropout"]
    config.pipeline_version = GPT2WordTagger.pipeline_version
    for name in ("bos_token_id", "pad_token_id", "eos_token_id"):
        setattr(config, name, getattr(tokenizer, name))
    config.use_cache = False
    if spec["max_length"] > config.max_position_embeddings:
        raise ValueError("Configured sequence length exceeds model context")
    model = GPT2WordTagger.from_pretrained(spec["repo_id"], revision=spec["revision"], config=config,
                                         cache_dir=str(PROJECT / "data/hf-cache"))
    assert model.config.vocab_size == len(tokenizer), "Tokenizer vocabulary must remain unchanged"
    assert all(parameter.requires_grad for parameter in model.parameters())
    return model


def make_trainer(system, model, run_dir, seed, training=False):
    spec = SYSTEMS[system]
    options: dict[str, Any] = dict(num_train_epochs=spec["epochs"], learning_rate=spec["learning_rate"],
        per_device_train_batch_size=spec["batch_size"], gradient_accumulation_steps=spec["gradient_accumulation"],
        weight_decay=spec["weight_decay"], warmup_ratio=spec["warmup_ratio"],
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model=EXPERIMENT["selection_metric"], greater_is_better=True, save_total_limit=1,
        gradient_checkpointing=spec.get("gradient_checkpointing", False),
        gradient_checkpointing_kwargs={"use_reentrant": False}) if training else {}
    cuda = torch.cuda.is_available()
    bf16 = cuda and torch.cuda.is_bf16_supported()
    args = TrainingArguments(output_dir=str(run_dir / ("checkpoints" if training else "evaluation")),
        per_device_eval_batch_size=spec["eval_batch_size"], seed=seed, data_seed=seed,
        fp16=cuda and not bf16, bf16=bf16, optim="adamw_torch", report_to="none", label_names=["labels"],
        remove_unused_columns=False, dataloader_num_workers=0, disable_tqdm=False, **options)
    return Trainer(model=model, args=args, processing_class=tokenizer, data_collator=collate,
        train_dataset=cast(TorchDataset[Any], model_data["train"]) if training else None,
        eval_dataset=cast(TorchDataset[Any], model_data["dev"]) if training else None,
        compute_metrics=metrics_for("dev" if training else "test"),
        preprocess_logits_for_metrics=reduce_logits)


def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 7. Training


In [ ]:
def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0]) if rows else []
    with Path(path).open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def package_versions():
    versions = {"python": platform.python_version()}
    for name in ("torch", "transformers", "datasets", "huggingface-hub", "numpy", "accelerate"):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

def run_identity(config, manifest, model_key, variant, seed):
    identity = {"configuration": config, "model": model_key, "variant": variant,
                "seed": seed, "data_manifest": manifest}
    if config.get("task_stage") == "upos_feats":
        identity["parent"] = parent_provenance(model_key, variant, seed)
    return identity

def model_experiment(config, model_key):
    orchestration = {"models", "active_systems", "seeds", "variants"}
    core = {key: value for key, value in config.items() if key not in orchestration}
    model = {key: value for key, value in config["models"][model_key].items() if key != "seeds"}
    return {**core, "models": {model_key: model}}


def verify_completed_run(run_dir, expected_identity):
    marker = json.loads((run_dir / "complete.json").read_text(encoding="utf-8"))
    if marker["experiment_sha256"] != sha256(run_dir / "experiment.json") or marker["test_results_sha256"] != sha256(run_dir / "test_results.json"):
        raise ValueError(f"Completed run checksum mismatch: {run_dir}")
    saved = json.loads((run_dir / "experiment.json").read_text(encoding="utf-8"))
    model_key = expected_identity["model"]
    actual = {**saved, "configuration": model_experiment(saved["configuration"], model_key)}
    expected = {**expected_identity, "configuration": model_experiment(expected_identity["configuration"], model_key)}
    if actual != expected:
        raise ValueError(f"Run configuration or corpus differs: {run_dir}")


def register_experiment(root, settings):
    path = root / "experiment.json"
    if path.exists():
        saved = json.loads(path.read_text(encoding="utf-8"))
        plan_keys = {"models", "active_systems", "seeds", "variants"}
        old_core = {key: value for key, value in saved.items() if key not in plan_keys}
        new_core = {key: value for key, value in settings.items() if key not in plan_keys}
        assert old_core == new_core, f"Configuration mismatch: {root}"
        for model_key, spec in settings["models"].items():
            if not any(p.is_file() for p in (root / model_key).rglob("*")):
                continue
            previous = saved["models"].get(model_key)
            assert previous is not None, f"Missing provenance for saved model: {model_key}"
            previous = {key: value for key, value in previous.items() if key != "seeds"}
            requested = {key: value for key, value in spec.items() if key != "seeds"}
            assert previous == requested, f"Configuration mismatch for saved model: {model_key}"
        if saved != settings:
            write_json(path, settings)
        return settings
    root.mkdir(parents=True, exist_ok=True)
    write_json(path, settings)
    return settings


def run_directory(system, seed):
    spec = SYSTEMS[system]
    return EXPERIMENT_DIR / spec["model"] / spec["variant"] / f"seed_{seed}"


def run_plan(config, project):
    return [{"model": SYSTEMS[system]["model"], "variant": SYSTEMS[system]["variant"], "seed": seed,
             "directory": str(Path(config["output_dir"]) / SYSTEMS[system]["model"] / SYSTEMS[system]["variant"] / f"seed_{seed}")}
            for system in config["active_systems"]
            for seed in config["models"][SYSTEMS[system]["model"]].get("seeds", config["seeds"])]


def model_weights_path(system, directory):
    return directory / "model.safetensors"


def saved_run_info(system, seed):
    run_dir = run_directory(system, seed)
    info = json.loads((run_dir / "training_info.json").read_text(encoding="utf-8"))
    model_key = SYSTEMS[system]["model"]
    assert model_experiment(info["experiment"], model_key) == model_experiment(experiment, model_key)
    assert info["system"] == system and info["seed"] == seed
    assert info["training_args"]["metric_for_best_model"] == EXPERIMENT["selection_metric"]
    assert model_weights_path(system, run_dir / "best_model").exists(), "Missing saved model"
    if TASK_STAGE == "upos_feats":
        assert info["parent"] == parent_provenance(SYSTEMS[system]["model"], SYSTEMS[system]["variant"], seed)
    return info


_parent_cache = {}


def parent_provenance(model_key, variant, seed):
    directory = UPOS_DIR / model_key / variant / f"seed_{seed}"
    paths = [directory / name for name in ("best_model/model.safetensors", "training_info.json",
                                           "experiment.json", "complete.json", "test_results.json")]
    if not all(path.is_file() for path in paths):
        raise RuntimeError(f"Complete the matching UPOS training and evaluation first: {directory}")
    signature = tuple((str(path), path.stat().st_size, path.stat().st_mtime_ns) for path in paths)
    if signature not in _parent_cache:
        expected = run_identity(UPOS_EXPERIMENT, data_manifest, model_key, variant, seed)
        verify_completed_run(directory, expected)
        info = json.loads(paths[1].read_text(encoding="utf-8"))
        assert model_experiment(info["experiment"], model_key) == model_experiment(
            {**UPOS_EXPERIMENT, "data_manifest": data_manifest}, model_key)
        assert info["seed"] == seed and info["system"] == f"{model_key}_{variant}"
        assert info["selection_metric"] == UPOS_EXPERIMENT["selection_metric"]
        _parent_cache[signature] = {"directory": str(directory.relative_to(PROJECT)),
            "weights_sha256": sha256(paths[0]), "training_info_sha256": sha256(paths[1]),
            "experiment_sha256": sha256(paths[2])}
    return _parent_cache[signature]


def train_run(system, seed):
    spec = SYSTEMS[system]
    run_dir = run_directory(system, seed)
    info_path, model_path = run_dir / "training_info.json", run_dir / "best_model"
    if info_path.exists() != model_weights_path(system, model_path).exists():
        raise RuntimeError(f"Incomplete saved run: {run_dir}. Restore its files or choose a new output directory.")
    if info_path.exists():
        saved_run_info(system, seed)
        print(f"Reusing {system}, seed {seed}")
        return
    set_seed(seed)
    prepare_system(system)
    parent = None
    if TASK_STAGE == "upos_feats":
        provenance = parent_provenance(spec["model"], spec["variant"], seed)
        parent = PROJECT / provenance["directory"] / "best_model"
    model = make_model(system, parent=parent)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    trainer = make_trainer(system, model, run_dir, seed, training=True)
    assert trainer.args.metric_for_best_model == EXPERIMENT["selection_metric"]
    print(f"{spec['name']} | seed {seed}; trainable parameters: {trainable:,}; selection: {EXPERIMENT['selection_metric']}")
    started_at, start = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    trainer.train()
    training_seconds = time.perf_counter() - start
    best_metric = trainer.state.best_metric
    if best_metric is None or not np.isfinite(best_metric):
        raise RuntimeError("Training finished without a finite development selection score")
    trainer.save_model(str(model_path))
    pd.DataFrame(trainer.state.log_history).to_csv(run_dir / "training_history.csv", index=False)
    info = {"system": system, "model": spec["model"], "variant": spec["variant"], "seed": seed,
        "experiment": experiment, "started_at_utc": started_at, "packages": package_versions(),
        "parameters_trainable": trainable, "selection_metric": EXPERIMENT["selection_metric"],
        "best_dev_score": float(best_metric), "best_checkpoint": trainer.state.best_model_checkpoint,
        "source_checkpoint": spec["repo_id"], "source_revision": spec["revision"],
        "training_seconds": training_seconds, "training_args": trainer.args.to_dict()}
    if parent is not None:
        info["parent"] = provenance
    write_json(info_path, info)
    write_json(run_dir / "experiment.json", run_identity(EXPERIMENT, data_manifest, spec["model"], spec["variant"], seed))
    del trainer, model
    release_memory()


def write_run_plan():
    completed = {(system, seed): all(path.is_file() for path in
        [run_directory(system, seed) / "training_info.json", model_weights_path(system, run_directory(system, seed) / "best_model")])
        for system in ACTIVE_SYSTEMS for seed in SYSTEMS[system].get("seeds", SEEDS)}
    write_json(EXPERIMENT_DIR / "run_plan.json", {"active_systems": ACTIVE_SYSTEMS,
        "model_seeds": {model: spec.get("seeds", SEEDS) for model, spec in MODELS.items()},
        "saved_runs": sum(completed.values()), "training_runs": sum(not done for done in completed.values())})
    return completed


if EncoderTagger.pipeline_version != UPOS_EXPERIMENT["schema"]:
    raise RuntimeError("Outdated model definition: restart the kernel and run all cells in order")
experiment = register_experiment(EXPERIMENT_DIR, {**EXPERIMENT, "data_manifest": data_manifest})
completed = write_run_plan()
print(f"{TASK_STAGE}: {sum(completed.values())} saved runs; {sum(not done for done in completed.values())} missing runs.")
for system in ACTIVE_SYSTEMS:
    for seed in SYSTEMS[system].get("seeds", SEEDS):
        train_run(system, seed)
write_run_plan()


## 8. Evaluation


In [ ]:
for seed in SEEDS:
    for system in ACTIVE_SYSTEMS:
        if seed not in SYSTEMS[system].get("seeds", SEEDS):
            continue
        assert (run_directory(system, seed) / "training_info.json").exists(), "Complete all configured training runs first"

run_results, prediction_frames = [], {}
for seed in SEEDS:
    for system in ACTIVE_SYSTEMS:
        if seed not in SYSTEMS[system].get("seeds", SEEDS):
            continue
        run_dir, spec = run_directory(system, seed), SYSTEMS[system]
        info = saved_run_info(system, seed)
        prepare_system(system)
        model = make_model(system, run_dir / "best_model")
        trainer = make_trainer(system, model, run_dir, seed)
        output = trainer.predict(cast(TorchDataset[Any], model_data["test"]))
        assert output.metrics is not None, "Missing test metrics"
        predictions, predicted_feats = decode_output(output.predictions, output.label_ids)
        scores, per_tag, rows = evaluate_sentences(sentences["test"], predictions, predicted_feats)
        for key in metric_names:
            assert math.isclose(scores[key], output.metrics[f"test_{key}"])
        inference_seconds = float(output.metrics["test_runtime"])
        scores.update(test_inference_seconds=inference_seconds,
                      test_words_per_second=scores["words"] / inference_seconds)
        frame = pd.DataFrame(rows)
        frame.to_csv(run_dir / "predictions.csv", index=False)
        wrong = ~frame.correct if predicted_feats is None else ~frame.joint_correct
        frame[wrong].to_csv(run_dir / "errors.csv", index=False)
        pd.DataFrame(per_tag).to_csv(run_dir / "per_tag_metrics.csv", index=False)
        if predicted_feats is not None:
            gold_features = [values for sentence in sentences["test"] for values in sentence["feats"]]
            guessed_features = [values for sentence in predicted_feats for values in sentence]
            feature_scores(gold_features, guessed_features).to_csv(run_dir / "per_feature_metrics.csv", index=False)
            attribute_scores(gold_features, guessed_features).to_csv(run_dir / "per_attribute_metrics.csv", index=False)
        write_json(run_dir / "test_results.json", scores)
        write_json(run_dir / "complete.json", {"completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "experiment_sha256": sha256(run_dir / "experiment.json"), "test_results_sha256": sha256(run_dir / "test_results.json")})
        run_results.append({"stage": TASK_STAGE, "system": system, "model": spec["model"], "variant": spec["variant"], "seed": seed,
                            **scores, "training_seconds": info["training_seconds"], "parameters_trainable": info["parameters_trainable"]})
        prediction_frames[(seed, system)] = frame
        del trainer, model, output
        release_memory()
results = pd.DataFrame(run_results)
results.to_csv(EXPERIMENT_DIR / "results.csv", index=False)
display(results[["system", "seed", *metric_names]].round(5))


## 9. Comparison

In [ ]:
def summarize(config, project):
    completed = [run for run in run_plan(config, project) if (Path(project) / run["directory"] / "complete.json").exists()]
    if not completed:
        raise RuntimeError("No completed runs: summary tables cannot be generated before training")
    _, manifest = load_corpus(config, project)
    rows = []
    for run in completed:
        directory = Path(project) / run["directory"]
        verify_completed_run(directory, run_identity(config, manifest, run["model"], run["variant"], run["seed"]))
        scores = json.loads((directory / "test_results.json").read_text(encoding="utf-8"))
        info = json.loads((directory / "training_info.json").read_text(encoding="utf-8"))
        rows.append({"stage": config.get("task_stage", "upos"), "system": f"{run['model']}_{run['variant']}",
                     "model": run["model"], "variant": run["variant"], "seed": run["seed"],
                     **scores, "training_seconds": info["training_seconds"], "parameters_trainable": info["parameters_trainable"]})
    root = Path(project) / config["output_dir"]
    fields = list(dict.fromkeys(key for row in rows for key in row))
    write_csv(root / "results.csv", rows, fields)
    metric_keys = (*metric_names, "training_seconds", "test_words_per_second")
    summary, changes = [], []
    for model in config["models"]:
        for variant in config["variants"]:
            group = [row for row in rows if row["model"] == model and row["variant"] == variant]
            if not group:
                continue
            entry = {"model": model, "variant": variant, "completed_seeds": len(group),
                     "planned_seeds": len(config["models"][model].get("seeds", config["seeds"]))}
            for key in metric_keys:
                if any(key not in row for row in group):
                    continue
                values = [row[key] for row in group]
                entry[f"{key}_mean"] = statistics.mean(values)
                entry[f"{key}_std"] = statistics.stdev(values) if len(values) > 1 else ""
            summary.append(entry)
        for seed in config["models"][model].get("seeds", config["seeds"]):
            pair = {row["variant"]: row for row in rows if row["model"] == model and row["seed"] == seed}
            if not {"original", "lowercase"}.issubset(pair):
                continue
            changes.append({"model": model, "seed": seed, **{
                f"{key}_delta_pp": 100 * (pair["lowercase"][key] - pair["original"][key])
                for key in metric_names}})
    write_csv(root / "comparison_mean_std.csv", summary, list(dict.fromkeys(key for row in summary for key in row)))
    case_ablation = {"original", "lowercase"}.issubset(config["variants"])
    if case_ablation:
        write_csv(root / "ablation_paired_deltas.csv", changes,
                  ["model", "seed", *[f"{key}_delta_pp" for key in metric_names]])
    ablation_summary = []
    for model in config["models"]:
        group = [row for row in changes if row["model"] == model]
        if not group:
            continue
        entry = {"model": model, "paired_seeds": len(group),
                 "planned_seeds": len(config["models"][model].get("seeds", config["seeds"]))}
        for key in metric_names:
            values = [row[f"{key}_delta_pp"] for row in group]
            entry[f"{key}_delta_pp_mean"] = statistics.mean(values)
            entry[f"{key}_delta_pp_std"] = statistics.stdev(values) if len(values) > 1 else ""
        ablation_summary.append(entry)
    if case_ablation:
        write_csv(root / "ablation_mean_std.csv", ablation_summary,
                  ["model", "paired_seeds", "planned_seeds", *[f"{key}_delta_pp_{stat}" for key in metric_names for stat in ("mean", "std")]])
    write_json(root / "summary_status.json", {"completed_runs": len(rows), "planned_runs": len(run_plan(config, project)),
                                              "complete": len(rows) == len(run_plan(config, project)),
                                              "delta_definition": ("lowercase minus original" if case_ablation else
                                                  "joint stage minus matching UPOS parent") + ", percentage points"})
    return {"completed_runs": len(rows), "planned_runs": len(run_plan(config, project))}


summarize(EXPERIMENT, PROJECT)
summary = results.groupby("system")[metric_names].agg(["mean", "std"]).reindex(ACTIVE_SYSTEMS) * 100
print("Mean and sample standard deviation (%), across three seeds for every model.")
display(summary.round(3))
comparison = results.groupby("system")[metric_names].mean().reindex(ACTIVE_SYSTEMS).reset_index()
comparison["system"] = comparison["system"].map({key: value["name"] for key, value in SYSTEMS.items()})
comparison.to_csv(EXPERIMENT_DIR / "comparison.csv", index=False)
display(comparison.set_index("system").mul(100).round(3))
paired = results.pivot(index="seed", columns="system", values=metric_names).reindex(columns=ACTIVE_SYSTEMS, level="system") * 100
paired.to_csv(EXPERIMENT_DIR / "paired_metrics.csv")
display(paired.round(3))
if {"original", "lowercase"}.issubset(EXPERIMENT["variants"]):
    display(pd.read_csv(EXPERIMENT_DIR / "ablation_paired_deltas.csv").round(3))


if TASK_STAGE == "upos_feats":
    stage_changes = []
    for system in ACTIVE_SYSTEMS:
        spec = SYSTEMS[system]
        for seed in spec.get("seeds", SEEDS):
            parent_provenance(spec["model"], spec["variant"], seed)
            before = json.loads((UPOS_DIR / spec["model"] / spec["variant"] / f"seed_{seed}" / "test_results.json").read_text(encoding="utf-8"))
            after = results.loc[(results.system == system) & (results.seed == seed)].iloc[0]
            stage_changes.append({"model": spec["model"], "variant": spec["variant"], "seed": seed,
                **{f"{key}_delta_pp": 100 * (after[key] - before[key]) for key in upos_metric_names}})
    stage_deltas = pd.DataFrame(stage_changes)
    stage_deltas.to_csv(EXPERIMENT_DIR / "stage_paired_deltas.csv", index=False)
    print("UPOS change after 10 additional epochs with FEATS (not an isolated causal FEATS effect), percentage points.")
    display(stage_deltas.round(3))
    stage_summary = stage_deltas.groupby(["model", "variant"])[[f"{key}_delta_pp" for key in upos_metric_names]].agg(["mean", "std"])
    stage_summary.to_csv(EXPERIMENT_DIR / "stage_mean_std.csv")
    display(stage_summary.round(3))


## 10. Errors


In [ ]:
error_rows = []
example_rows = []
for system in ACTIVE_SYSTEMS:
    counts = Counter()
    for seed in SYSTEMS[system].get("seeds", SEEDS):
        frame = prediction_frames[(seed, system)]
        wrong = frame[frame.true_label != frame.predicted_label]
        counts.update(zip(wrong.true_label, wrong.predicted_label))
        for row in wrong.head(3).to_dict("records"):
            words = sentences["test"][row["sentence_index"]]["tokens"]
            example_rows.append({"system": system, "seed": seed, **row, "sentence": " ".join(words)})
    error_rows.extend({"system": system, "true_tag": a, "predicted_tag": b, "errors": count}
                      for (a, b), count in counts.most_common(3))
error_table = pd.DataFrame(error_rows)
print("Top three UPOS confusions per system, aggregated across seeds.")
display(error_table)
error_table.to_csv(EXPERIMENT_DIR / "error_summary.csv", index=False)
error_examples = pd.DataFrame(example_rows)
display(error_examples)
error_examples.to_csv(EXPERIMENT_DIR / "error_examples.csv", index=False)


if TASK_STAGE == "upos_feats":
    feature_errors = []
    for system in ACTIVE_SYSTEMS:
        frames = [pd.read_csv(run_directory(system, seed) / "per_feature_metrics.csv")
                  for seed in SYSTEMS[system].get("seeds", SEEDS)]
        table = pd.concat(frames).groupby("feature")[["fp", "fn"]].sum()
        table["errors"] = table.fp + table.fn
        top = table.nlargest(5, "errors").reset_index()
        top.insert(0, "system", system)
        feature_errors.append(top)
    feature_error_table = pd.concat(feature_errors, ignore_index=True)
    print("Top five FEATS pair errors per system (false positives + false negatives across seeds).")
    display(feature_error_table)
    feature_error_table.to_csv(EXPERIMENT_DIR / "feature_error_summary.csv", index=False)
